In [1]:
########################################################
###################### Importierte Packete #############
########################################################

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rc                                                   # To use a LaTeX font
from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from matplotlib.colors import rgb2hex

#import csv

import numpy as np

import pandas as pd

from IPython.display import display, HTML

import scipy
from scipy import stats
import scipy.constants as sc
from scipy import ndimage
import scipy.signal
from scipy.interpolate import interp1d

import statsmodels.api as sm

from mpl_toolkits import mplot3d                                            # Um 3D Plots zu machen

import os

##### Saving/packing of data
import pickle

##### To create colour gradients
from colour import Color

In [2]:
# importing the own functions from the scientific_utils package
from scientific_utils.general.analysis import Add_Columns_To_Dataframe

In [3]:
#########################################################
##################### Start of the code ##################
#########################################################

In [4]:
########## Open the saved lists from the .pkl files ########

##### 
with open('../data/Batt15/processed/dataframes_of_charge_discharge_capacities_per_cycle.pkl', 'rb') as f:
    dataframes_of_charge_discharge_capacities_per_cycle = pickle.load(f)    

##### 
with open('../data/Batt15/processed/dataframes_of_potential_drop_beginning_OCV_and_end_OCV_per_cycle.pkl', 'rb') as f:
    dataframes_of_potential_drop_beginning_OCV_and_end_OCV_per_cycle = pickle.load(f) 

##### 
with open('../data/Batt15/processed/dataframes_of_potential_drop_between_CP_and_OCV_per_cycle.pkl', 'rb') as f:
    dataframes_of_potential_drop_between_CP_and_OCV_per_cycle = pickle.load(f) 
 
#### 
with open('../data/Batt15/processed/dataframes_of_dQdV_CP_charge_peaks_cycling.pkl', 'rb') as f:
    dataframes_of_dQdV_CP_charge_peaks_cycling = pickle.load(f) 
    
#### 
with open('../data/Batt15/processed/dataframes_of_dQdV_CP_discharge_peaks_cycling.pkl', 'rb') as f:
    dataframes_of_dQdV_CP_discharge_peaks_cycling = pickle.load(f) 


In [5]:
########## Define colours by hex code ##########
yellow = '#ffa200'
orange = '#f97625'
lightpink = '#e94a3a'
pink = '#ce1b49'
lightpurple = '#a90054'
purple = '#7e0059'
darkpurple = '#4c0057'
darkblue = '#0a004d'

########## Colours in a list to access them in a for loop ##########
colors = [yellow, orange, lightpink, pink, lightpurple, purple, darkpurple, darkblue]


########## Colour gradient with the colour package
#beginnig_color = Color("yellow")
#end_color = Color("blue")
#number_of_colorpoints = 4
#color_gradient = list(beginnig_color.range_to(end_color,number_of_colorpoints))
#print(color_gradient)


########## Cologradient with Matplotlib Colormaps
# Available colormaps: https://matplotlib.org/3.1.0/tutorials/colors/colormaps.html#ibm
number_of_colorpoints = 10 # number of colour points taken from the colour gradient with equal distance
colormap = plt.get_cmap('RdBu', number_of_colorpoints) # Loads a ready-made colormap from Matplotlib
array_of_RGBAcolors = colormap(range(number_of_colorpoints)) # Returns the colormap as RGBA values scaled from 0 to 1
list_of_Hexcolors = [ rgb2hex(array_of_RGBAcolors[i,:]) for i in range(array_of_RGBAcolors.shape[0]) ] # Converts the scaled RGBA values to hex code and puts them in a list


In [6]:
########## Font settings for the PGF file (used in LaTeX) ##########
pgf_with_latex = {                      # setup matplotlib to use latex for output
    "pgf.texsystem": "xelatex",         # change this if using xetex or lautex
    "text.usetex": False,                # use LaTeX to write all text
    "pgf.rcfonts": False,               # don't setup fonts from rc parameters
    "axes.labelsize": 12,               # LaTeX default is 10pt font.
    "legend.fontsize": 10,              # Make the legend/label fonts a little smaller
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "pgf.preamble": "\n".join([
        r"\usepackage[utf8x]{inputenc}",    # use utf8 fonts becasue your computer can handle it :)
        r"\usepackage[T1]{fontenc}",        # plots will be generated using this preamble
        ])
    }

##### one style for all plots: the shared publication style (scientific_utils.general.plotting) replaces the local settings above
from scientific_utils.general.plotting import pgf_with_latex
plt.rcParams.update(pgf_with_latex)

In [7]:
###################################### Definition of Function ########################################

In [8]:
########## To delete the file with the general data of all batteries and save it again
    
dataframe_with_all_extracted_data_from_Cycling = pd.DataFrame() # creating new empty dataframe
#dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging.append(pd.Series(name=0))

#print(dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging)
    
##### General_Data_from_every_Cell
with open('../data/Batt15/processed/dataframe_with_all_extracted_data_from_Cycling.pkl', 'wb') as f:
    pickle.dump(dataframe_with_all_extracted_data_from_Cycling, f)

In [9]:
########## Saving the dQdV peaks of discharge and charge steps in the dataframe ##########
new_columns = {} # the values are collected first and added at once (adding the columns one by one fragments the dataframe)
### charging steps of the cycling dQdV value of the dQdV peak
labels_dQdV_charge_peaks = []
for i in range(0, len(dataframes_of_dQdV_CP_charge_peaks_cycling)):
    label = 'Cycle %s dQdV peak charge / mAh/V' %(i+1)
    labels_dQdV_charge_peaks.append(label)
   
for i in range(0, len(dataframes_of_dQdV_CP_charge_peaks_cycling)):
    new_columns[labels_dQdV_charge_peaks[i]] = dataframes_of_dQdV_CP_charge_peaks_cycling.iloc[i]['Peak Savgol-Filter dQ/dV / mAh/V'] # The value is saved in the dictionary under the column name; Add_Columns_To_Dataframe replaces existing columns and adds new ones

### discharging steps of the rate test dQdV value of the dQdV peak
labels_dQdV_discharge_peaks = []
for i in range(0, len(dataframes_of_dQdV_CP_discharge_peaks_cycling)):
    label = 'Cycle %s dQdV peak discharge / mAh/V' %(i+1)
    labels_dQdV_discharge_peaks.append(label)
    
for i in range(0, len(dataframes_of_dQdV_CP_discharge_peaks_cycling)):
    new_columns[labels_dQdV_discharge_peaks[i]] = dataframes_of_dQdV_CP_discharge_peaks_cycling.iloc[i]['Peak Savgol-Filter dQ/dV / mAh/V']

dataframe_with_all_extracted_data_from_Cycling = Add_Columns_To_Dataframe(dataframe_with_all_extracted_data_from_Cycling, new_columns)

#print(dataframe_with_all_extracted_data_from_Cycling)

In [10]:
########## Saving the dQdV peaks of discharge and charge steps in the dataframe ##########
new_columns = {} # the values are collected first and added at once (adding the columns one by one fragments the dataframe)
### charging steps of the cycling curve potential of the dQdV peak
labels_dQdV_charge_peaks = []
for i in range(0, len(dataframes_of_dQdV_CP_charge_peaks_cycling)):
    label = 'Cycle %s dQdV peak charge / V' %(i+1)
    labels_dQdV_charge_peaks.append(label)
   
for i in range(0, len(dataframes_of_dQdV_CP_charge_peaks_cycling)):
    new_columns[labels_dQdV_charge_peaks[i]] = dataframes_of_dQdV_CP_charge_peaks_cycling.iloc[i]['Peak Savgol-Filter Voltage (V)'] # The value is saved in the dictionary under the column name; Add_Columns_To_Dataframe replaces existing columns and adds new ones

### discharging steps of the rate test dQdV value of the dQdV peak
labels_dQdV_discharge_peaks = []
for i in range(0, len(dataframes_of_dQdV_CP_discharge_peaks_cycling)):
    label = 'Cycle %s dQdV peak discharge / V' %(i+1)
    labels_dQdV_discharge_peaks.append(label)
    
for i in range(0, len(dataframes_of_dQdV_CP_discharge_peaks_cycling)):
    new_columns[labels_dQdV_discharge_peaks[i]] = dataframes_of_dQdV_CP_discharge_peaks_cycling.iloc[i]['Peak Savgol-Filter Voltage (V)']

dataframe_with_all_extracted_data_from_Cycling = Add_Columns_To_Dataframe(dataframe_with_all_extracted_data_from_Cycling, new_columns)

#print(dataframe_with_all_extracted_data_from_Cycling)

In [11]:
########## Saving the total charge and discharge capaciteies in the dataframe ##########
new_columns = {} # the values are collected first and added at once (adding the columns one by one fragments the dataframe)
### charging steps of cycling
labels_charge_capacities = [] # List of labels used as column names of the dataframe
for i in range(0, len(dataframes_of_charge_discharge_capacities_per_cycle)):
    label = 'Cycle %s total charge capacity / mAh' %(i+1)
    labels_charge_capacities.append(label)

for i in range(0, len(dataframes_of_charge_discharge_capacities_per_cycle)):
    new_columns[labels_charge_capacities[i]] = dataframes_of_charge_discharge_capacities_per_cycle.iloc[i]['Charge Capacity / mAh'] # The value is saved in the dictionary under the column name; Add_Columns_To_Dataframe replaces existing columns and adds new ones

### discharging steps of cycling
labels_discharge_capacities = [] # List of labels used as column names of the dataframe
for i in range(0, len(dataframes_of_charge_discharge_capacities_per_cycle)):
    label = 'Cycle %s total discharge capacity / mAh' %(i+1)
    labels_discharge_capacities.append(label)

for i in range(0, len(dataframes_of_charge_discharge_capacities_per_cycle)):
    new_columns[labels_discharge_capacities[i]] = dataframes_of_charge_discharge_capacities_per_cycle.iloc[i]['Discharge Capacity / mAh']

dataframe_with_all_extracted_data_from_Cycling = Add_Columns_To_Dataframe(dataframe_with_all_extracted_data_from_Cycling, new_columns)

#print(dataframe_with_all_extracted_data_from_Cycling)

In [12]:
########## Saving the total charge and discharge capaciteies in the dataframe ##########
new_columns = {} # the values are collected first and added at once (adding the columns one by one fragments the dataframe)
### charging steps of the rate test
labels_charge_potential_drop_CP_to_OCV = [] # List of labels used as column names of the dataframe
for i in range(0, len(dataframes_of_charge_discharge_capacities_per_cycle)):
    label = 'Cycle %s DeltaV CP charge to OCV / V' %(i+1)
    labels_charge_potential_drop_CP_to_OCV.append(label)

for i in range(0, len(dataframes_of_potential_drop_between_CP_and_OCV_per_cycle)):
    new_columns[labels_charge_potential_drop_CP_to_OCV[i]] = dataframes_of_potential_drop_between_CP_and_OCV_per_cycle.iloc[i]['DeltaV CP charge to OCV'] # The value is saved in the dictionary under the column name; Add_Columns_To_Dataframe replaces existing columns and adds new ones

### discharging steps of the rate test
labels_discharge_potential_drop_CP_to_OCV = [] # List of labels used as column names of the dataframe
for i in range(0, len(dataframes_of_charge_discharge_capacities_per_cycle)):
    label = 'Cycle %s DeltaV CP discharge to OCV / V' %(i+1)
    labels_discharge_potential_drop_CP_to_OCV.append(label)

for i in range(0, len(dataframes_of_potential_drop_between_CP_and_OCV_per_cycle)):
    new_columns[labels_discharge_potential_drop_CP_to_OCV[i]] = dataframes_of_potential_drop_between_CP_and_OCV_per_cycle.iloc[i]['DeltaV CP discharge to OCV']

dataframe_with_all_extracted_data_from_Cycling = Add_Columns_To_Dataframe(dataframe_with_all_extracted_data_from_Cycling, new_columns)

#print(dataframe_with_all_extracted_data_from_Cycling)

In [13]:
########## Saving the potential difference between the beginning of the ocv step and after trhee hours##########
new_columns = {} # the values are collected first and added at once (adding the columns one by one fragments the dataframe)
### OCV after charging steps of the rate test
labels_charge_potential_drop_OCV = [] # List of labels used as column names of the dataframe
for i in range(0, len(dataframes_of_charge_discharge_capacities_per_cycle)):
    label = 'Cycle %s charge; DeltaV OCV after three hours / V' %(i+1)
    labels_charge_potential_drop_OCV.append(label)

for i in range(0, len(dataframes_of_potential_drop_beginning_OCV_and_end_OCV_per_cycle)):
    new_columns[labels_charge_potential_drop_OCV[i]] = dataframes_of_potential_drop_beginning_OCV_and_end_OCV_per_cycle.iloc[i]['Potential Difference OCV charge at end'] # The value is saved in the dictionary under the column name; Add_Columns_To_Dataframe replaces existing columns and adds new ones

### OCV after discharging steps of the rate test
labels_discharge_potential_drop_OCV = [] # List of labels used as column names of the dataframe
for i in range(0, len(dataframes_of_charge_discharge_capacities_per_cycle)):
    label = 'Cycle %s discharge; DeltaV OCV after three hours / V' %(i+1)
    labels_discharge_potential_drop_OCV.append(label)
    
for i in range(0, len(dataframes_of_potential_drop_beginning_OCV_and_end_OCV_per_cycle)):
    new_columns[labels_discharge_potential_drop_OCV[i]] = dataframes_of_potential_drop_beginning_OCV_and_end_OCV_per_cycle.iloc[i]['Potential Difference OCV discharge at end']

dataframe_with_all_extracted_data_from_Cycling = Add_Columns_To_Dataframe(dataframe_with_all_extracted_data_from_Cycling, new_columns)

#print(dataframe_with_all_extracted_data_from_Cycling)

In [14]:
display(dataframe_with_all_extracted_data_from_Cycling)

,Cycle 1 dQdV peak charge / mAh/V,Cycle 2 dQdV peak charge / mAh/V,Cycle 3 dQdV peak charge / mAh/V,Cycle 4 dQdV peak charge / mAh/V,Cycle 5 dQdV peak charge / mAh/V,Cycle 6 dQdV peak charge / mAh/V,Cycle 7 dQdV peak charge / mAh/V,Cycle 8 dQdV peak charge / mAh/V,Cycle 9 dQdV peak charge / mAh/V,Cycle 10 dQdV peak charge / mAh/V,...,Cycle 40 discharge; DeltaV OCV after three hours / V,Cycle 41 discharge; DeltaV OCV after three hours / V,Cycle 42 discharge; DeltaV OCV after three hours / V,Cycle 43 discharge; DeltaV OCV after three hours / V,Cycle 44 discharge; DeltaV OCV after three hours / V,Cycle 45 discharge; DeltaV OCV after three hours / V,Cycle 46 discharge; DeltaV OCV after three hours / V,Cycle 47 discharge; DeltaV OCV after three hours / V,Cycle 48 discharge; DeltaV OCV after three hours / V,Cycle 49 discharge; DeltaV OCV after three hours / V
0,2.624263,2.90298,2.886462,2.855946,2.84321,2.823015,2.796804,2.772527,2.746459,2.723416,...,-0.8706,-0.876,-0.8861,-0.8962,-0.9048,-0.9142,-0.9248,-0.9374,-0.9493,-0.9618


In [15]:
########## Save the data ##########
with open('../data/Batt15/processed/dataframe_with_all_extracted_data_from_Cycling.pkl', 'wb') as f:
    pickle.dump(dataframe_with_all_extracted_data_from_Cycling, f)